# FedVocabHAR WISDM Phase Zero

This notebook makes the identity-mechanism decision before the full federated pipeline is implemented. It compares:

1. calibrated soft-label summaries;
2. prototypes from a frozen shared reference encoder; and
3. a dual signature combining both views.

Development activities train and calibrate the pair matcher. Final novel activities and held-out subjects are used only once for evaluation. Local clusters are oracle-defined in this phase so detector errors cannot hide identity-resolution failures.

**Go criteria:** dual-signature pair AUROC at least 0.85, and registry birth precision at least 0.90 with recall at least 0.70 under independent artifacts. Mixtures are evaluated with abstention and reported separately.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import copy, hashlib, json, math, os, random, shutil, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

@dataclass(frozen=True)
class Config:
    project: str = '/content/drive/MyDrive/FedVocabHAR'
    seed: int = 42
    window: int = 128
    stride: int = 64
    train_fraction: float = 0.60
    known: tuple = ('A','B','C','D','E','R')
    development_novel: tuple = ('F','G','H','I','K')
    final_novel: tuple = ('L','M','O','P','Q','S')
    development_subjects: tuple = tuple(range(1635,1642))
    final_subjects: tuple = tuple(range(1643,1651))
    batch_size: int = 256
    epochs: int = 30
    learning_rate: float = 1e-3
    embedding_dim: int = 64
    pair_target_fpr: float = 0.05
    registry_margin: float = 0.05
    registry_trials: int = 100

CFG = Config()
PROJECT = Path(CFG.project)
RAW = PROJECT / 'data' / 'WISDM-2019' / 'raw' / 'watch' / 'accel'
ARTIFACTS = PROJECT / 'artifacts' / 'phase0'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert RAW.exists(), f'WISDM path not found: {RAW}'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('project:', PROJECT)

device: cuda
project: /content/drive/MyDrive/FedVocabHAR


## Build a chronological WISDM window cache

Only the smartwatch accelerometer is used in Phase Zero. Each recording stays ordered, and no overlapping windows cross subject or activity boundaries. The cache is saved in Drive so later Colab sessions do not repeatedly parse the raw text files.

In [3]:
ACTIVITY_NAMES = {
    'A':'walking','B':'jogging','C':'stairs','D':'sitting','E':'standing','F':'typing',
    'G':'teeth','H':'soup','I':'chips','J':'pasta','K':'drinking','L':'sandwich',
    'M':'kicking','O':'catch','P':'dribbling','Q':'writing','R':'clapping','S':'folding'
}

def parse_subject(path):
    groups = defaultdict(list)
    with path.open('r', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            fields = line.strip().rstrip(';').split(',')
            if len(fields) != 6: continue
            try: groups[fields[1]].append([float(fields[3]),float(fields[4]),float(fields[5])])
            except ValueError: continue
    return {a:np.asarray(v,dtype=np.float32) for a,v in groups.items()}

def make_windows(samples, size, stride):
    windows=[]; positions=[]
    denominator=max(1,len(samples)-size)
    for start in range(0,len(samples)-size+1,stride):
        windows.append(samples[start:start+size].T)
        positions.append(start/denominator)
    return np.asarray(windows,dtype=np.float32), np.asarray(positions,dtype=np.float32)

def build_cache(cache_path):
    subjects = list(CFG.development_subjects)+list(CFG.final_subjects)
    wanted=set(CFG.known+CFG.development_novel+CFG.final_novel)
    xs=[]; subject_ids=[]; activities=[]; positions=[]
    for subject in subjects:
        groups=parse_subject(RAW/f'data_{subject}_accel_watch.txt')
        for activity,samples in groups.items():
            if activity not in wanted: continue
            w,p=make_windows(samples,CFG.window,CFG.stride)
            xs.append(w); positions.extend(p); subject_ids.extend([subject]*len(w)); activities.extend([activity]*len(w))
        print('parsed',subject)
    payload=dict(x=np.concatenate(xs),subject=np.asarray(subject_ids),activity=np.asarray(activities),position=np.asarray(positions))
    np.savez_compressed(cache_path,**payload)
    return payload

CACHE=ARTIFACTS/f'wisdm_watch_accel_w{CFG.window}_s{CFG.stride}.npz'
if CACHE.exists():
    loaded=np.load(CACHE,allow_pickle=False); DATA={k:loaded[k] for k in loaded.files}
else:
    DATA=build_cache(CACHE)
print({k:v.shape for k,v in DATA.items()})
print(pd.crosstab(DATA['subject'],DATA['activity']))

parsed 1635
parsed 1636
parsed 1637
parsed 1638
parsed 1639
parsed 1640
parsed 1641
parsed 1643
parsed 1644
parsed 1645
parsed 1646
parsed 1647
parsed 1648
parsed 1649
parsed 1650
{'x': (19704, 3, 128), 'subject': (19704,), 'activity': (19704,), 'position': (19704,)}
col_0    A    B    C    D    E    F    G    H    I    K    L    M    O    P  \
row_0                                                                         
1635    55   55   55   55   55   55   55   55   55   55   55   55   55   55   
1636    55   55   55   55   55   55   55   55   55   55   55   55   55   55   
1637   138  138  138  138  138  138  138  138  138  138  138  138  138  138   
1638   138  138  138  138  138  138  138  138  138  138  138  138  138  138   
1639   138  138  138  138  138  138  138  138  138  138    0  138  138  138   
1640   138  138  138  138  138  138  138  138  138  138  138  138  138  138   
1641    55   55   55   55   55   55   55   55   55   55   55   55   55   55   
1643    55   55   55 

## Train the frozen reference encoder

The reference model learns only the six known activities from development subjects. Final subjects and all novel activities remain excluded. Channel normalization is fitted on the known training windows only.

In [4]:
x=DATA['x']; subjects=DATA['subject']; activities=DATA['activity']; positions=DATA['position']
dev_mask=np.isin(subjects,CFG.development_subjects)
final_mask=np.isin(subjects,CFG.final_subjects)
known_mask=np.isin(activities,CFG.known)
train_mask=dev_mask & known_mask & (positions<CFG.train_fraction)
val_mask=dev_mask & known_mask & (positions>=CFG.train_fraction)
test_known_mask=final_mask & known_mask

channel_mean=x[train_mask].mean(axis=(0,2),keepdims=True)
channel_std=x[train_mask].std(axis=(0,2),keepdims=True)+1e-6
x_norm=(x-channel_mean)/channel_std
label_to_index={label:i for i,label in enumerate(CFG.known)}

def make_loader(mask,shuffle):
    xx=torch.from_numpy(x_norm[mask])
    yy=torch.tensor([label_to_index[a] for a in activities[mask]],dtype=torch.long)
    return DataLoader(TensorDataset(xx,yy),batch_size=CFG.batch_size,shuffle=shuffle,num_workers=2,pin_memory=True)

train_loader=make_loader(train_mask,True); val_loader=make_loader(val_mask,False); known_test_loader=make_loader(test_known_mask,False)
print('known train/validation/final-subject test:',train_mask.sum(),val_mask.sum(),test_known_mask.sum())

known train/validation/final-subject test: 2586 1716 2711


In [5]:
class ReferenceNet(nn.Module):
    def __init__(self,n_classes,embedding_dim):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv1d(3,64,7,padding=3),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2),
            nn.Conv1d(64,128,5,padding=2),nn.BatchNorm1d(128),nn.ReLU(),nn.MaxPool1d(2),
            nn.Conv1d(128,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
        self.embedding=nn.Linear(128,embedding_dim)
        self.head=nn.Linear(embedding_dim,n_classes)
    def forward(self,x,return_embedding=False):
        h=self.features(x).squeeze(-1)
        z=F.normalize(self.embedding(h),dim=1)
        logits=self.head(z)
        return (logits,z) if return_embedding else logits

model=ReferenceNet(len(CFG.known),CFG.embedding_dim).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG.learning_rate,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,CFG.epochs)

@torch.no_grad()
def accuracy(loader):
    model.eval(); correct=total=0
    for xb,yb in loader:
        pred=model(xb.to(DEVICE)).argmax(1).cpu(); correct+=(pred==yb).sum().item(); total+=len(yb)
    return correct/max(total,1)

best_state=None; best_val=-1
history=[]
for epoch in range(CFG.epochs):
    model.train(); losses=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(); loss=F.cross_entropy(model(xb),yb); loss.backward(); optimizer.step(); losses.append(loss.item())
    scheduler.step(); val_acc=accuracy(val_loader)
    history.append({'epoch':epoch+1,'loss':float(np.mean(losses)),'val_accuracy':val_acc})
    if val_acc>best_val: best_val=val_acc; best_state=copy.deepcopy(model.state_dict())
    print(epoch+1,history[-1])
model.load_state_dict(best_state)
known_test_accuracy=accuracy(known_test_loader)
torch.save({'state_dict':model.state_dict(),'config':asdict(CFG),'channel_mean':channel_mean,'channel_std':channel_std},ARTIFACTS/'reference_model.pt')
print('best development accuracy:',best_val,'held-out-subject known accuracy:',known_test_accuracy)

1 {'epoch': 1, 'loss': 1.6147380742159756, 'val_accuracy': 0.7074592074592074}
2 {'epoch': 2, 'loss': 1.4132972955703735, 'val_accuracy': 0.8403263403263403}
3 {'epoch': 3, 'loss': 1.3042065447027034, 'val_accuracy': 0.9184149184149184}
4 {'epoch': 4, 'loss': 1.2180458090522073, 'val_accuracy': 0.9335664335664335}
5 {'epoch': 5, 'loss': 1.1474760987541892, 'val_accuracy': 0.9364801864801865}
6 {'epoch': 6, 'loss': 1.0880482630296187, 'val_accuracy': 0.9498834498834499}
7 {'epoch': 7, 'loss': 1.031542181968689, 'val_accuracy': 0.9481351981351981}
8 {'epoch': 8, 'loss': 0.9806204438209534, 'val_accuracy': 0.9562937062937062}
9 {'epoch': 9, 'loss': 0.9324819066307761, 'val_accuracy': 0.967948717948718}
10 {'epoch': 10, 'loss': 0.8875986608591947, 'val_accuracy': 0.9801864801864801}
11 {'epoch': 11, 'loss': 0.8447444168004122, 'val_accuracy': 0.9685314685314685}
12 {'epoch': 12, 'loss': 0.8076282251964916, 'val_accuracy': 0.9644522144522144}
13 {'epoch': 13, 'loss': 0.7752317515286532, 'va

## Construct candidate signatures

Candidate prototypes are clipped before aggregation. In the full federated study, privacy noise will be added after clipping; Phase Zero first measures the unnoised ceiling.

In [6]:
@torch.no_grad()
def summarize_candidate(mask,clip_norm=1.0,temperature=1.0):
    model.eval(); xx=torch.from_numpy(x_norm[mask]); logits=[]; embeddings=[]
    for start in range(0,len(xx),CFG.batch_size):
        lg,z=model(xx[start:start+CFG.batch_size].to(DEVICE),return_embedding=True)
        logits.append(lg.cpu()); embeddings.append(z.cpu())
    logits=torch.cat(logits); embeddings=torch.cat(embeddings)
    probs=F.softmax(logits/temperature,dim=1).numpy(); z=embeddings.numpy()
    norms=np.linalg.norm(z,axis=1,keepdims=True)+1e-12; z=z*np.minimum(1.0,clip_norm/norms)
    prototype=z.mean(0); prototype=prototype/(np.linalg.norm(prototype)+1e-12)
    entropy=-(probs*np.log(probs+1e-12)).sum(1)/math.log(probs.shape[1])
    return {'p_mean':probs.mean(0),'p_std':probs.std(0),'prototype':prototype,
            'uncertainty':float(entropy.mean()),'support':int(len(probs))}

def build_candidates(subject_list,activity_list):
    output={}
    for subject in subject_list:
        for activity in activity_list:
            mask=(subjects==subject)&(activities==activity)
            if mask.sum(): output[(int(subject),str(activity))]=summarize_candidate(mask)
    return output

development_candidates=build_candidates(CFG.development_subjects,CFG.development_novel)
final_candidates=build_candidates(CFG.final_subjects,CFG.final_novel)
print('development/final candidates:',len(development_candidates),len(final_candidates))

development/final candidates: 35 48


## Learn and calibrate pair compatibility

The pair matcher is trained only on development activities. Cross-client subject pairs are separated into matcher-training and calibration groups. The final test uses different subjects and different novel activities.

In [7]:
def cosine(a,b): return float(np.dot(a,b)/((np.linalg.norm(a)*np.linalg.norm(b))+1e-12))

def pair_features(a,b,variant):
    scalar=np.array([abs(a['uncertainty']-b['uncertainty']),math.log1p(min(a['support'],b['support'])),
                     cosine(a['p_mean'],b['p_mean']),cosine(a['prototype'],b['prototype'])],dtype=np.float32)
    soft=np.concatenate([np.abs(a['p_mean']-b['p_mean']),a['p_mean']*b['p_mean'],
                         np.abs(a['p_std']-b['p_std']),scalar[:3]])
    proto=np.concatenate([np.abs(a['prototype']-b['prototype']),a['prototype']*b['prototype'],scalar[[0,1,3]]])
    return {'soft':soft,'prototype':proto,'dual':np.concatenate([soft,proto])}[variant]

def candidate_pairs(candidates,subject_list,activity_list,allowed_subject_pairs=None):
    rows=[]
    for i,left in enumerate(subject_list):
        for right in subject_list[i+1:]:
            key=tuple(sorted((int(left),int(right))))
            if allowed_subject_pairs is not None and key not in allowed_subject_pairs: continue
            for a in activity_list:
                for b in activity_list:
                    if (left,a) in candidates and (right,b) in candidates:
                        rows.append((candidates[(left,a)],candidates[(right,b)],int(a==b),left,right,a,b))
    return rows

rng=np.random.default_rng(CFG.seed)
subject_pairs=[(a,b) for i,a in enumerate(CFG.development_subjects) for b in CFG.development_subjects[i+1:]]
rng.shuffle(subject_pairs); split=max(1,int(.70*len(subject_pairs)))
matcher_pairs=set(subject_pairs[:split]); calibration_pairs=set(subject_pairs[split:])
train_pairs=candidate_pairs(development_candidates,list(CFG.development_subjects),list(CFG.development_novel),matcher_pairs)
cal_pairs=candidate_pairs(development_candidates,list(CFG.development_subjects),list(CFG.development_novel),calibration_pairs)
test_pairs=candidate_pairs(final_candidates,list(CFG.final_subjects),list(CFG.final_novel))

def ece(y,p,bins=10):
    edges=np.linspace(0,1,bins+1); value=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if mask.any(): value+=mask.mean()*abs(y[mask].mean()-p[mask].mean())
    return float(value)

models={}; thresholds={}; pair_results=[]
for variant in ('soft','prototype','dual'):
    Xtr=np.stack([pair_features(a,b,variant) for a,b,*_ in train_pairs]); ytr=np.array([r[2] for r in train_pairs])
    Xcal=np.stack([pair_features(a,b,variant) for a,b,*_ in cal_pairs]); ycal=np.array([r[2] for r in cal_pairs])
    Xte=np.stack([pair_features(a,b,variant) for a,b,*_ in test_pairs]); yte=np.array([r[2] for r in test_pairs])
    matcher=Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(max_iter=3000,class_weight='balanced',C=1.0))])
    matcher.fit(Xtr,ytr); pcal=matcher.predict_proba(Xcal)[:,1]; pte=matcher.predict_proba(Xte)[:,1]
    threshold=float(np.quantile(pcal[ycal==0],1-CFG.pair_target_fpr,method='higher'))
    models[variant]=matcher; thresholds[variant]=threshold
    pair_results.append({'variant':variant,'test_auroc':roc_auc_score(yte,pte),'test_average_precision':average_precision_score(yte,pte),
                         'test_brier':brier_score_loss(yte,pte),'test_ece':ece(yte,pte),'threshold':threshold,
                         'test_tpr':float((pte[yte==1]>=threshold).mean()),'test_fpr':float((pte[yte==0]>=threshold).mean())})
pair_results=pd.DataFrame(pair_results)
pair_results

,variant,test_auroc,test_average_precision,test_brier,test_ece,threshold,test_tpr,test_fpr
0,soft,0.531222,0.225918,0.495375,0.553270,0.614167,0.559524,0.601190
1,prototype,0.540781,0.170578,0.328829,0.322348,0.873440,0.178571,0.202381
2,dual,0.526396,0.231862,0.653086,0.657124,0.904786,0.636905,0.669048


## Simulate the provisional registry

The registry accepts at most one vote per client and uses complete-link compatibility. A report that has two plausible destinations within the ambiguity margin starts or remains in a provisional entry. Independent and mixed artifacts are reported separately.

In [8]:
def synthetic_candidate(subject,mode,rng):
    keys=[key for key in final_candidates if key[0]==subject]
    if mode=='mixture':
        k1,k2=rng.choice(len(keys),2,replace=False); a=final_candidates[keys[k1]]; b=final_candidates[keys[k2]]; w=rng.uniform(.25,.75)
        result={name:w*a[name]+(1-w)*b[name] for name in ('p_mean','p_std','prototype')}
        result['prototype']/=np.linalg.norm(result['prototype'])+1e-12
        result['uncertainty']=w*a['uncertainty']+(1-w)*b['uncertainty']; result['support']=min(a['support'],b['support'])
        return result
    dim=len(next(iter(final_candidates.values()))['prototype'])
    p=rng.dirichlet(np.ones(len(CFG.known)))
    z=rng.normal(size=dim); z/=np.linalg.norm(z)+1e-12
    return {'p_mean':p,'p_std':rng.uniform(0,.15,len(p)),'prototype':z,'uncertainty':rng.uniform(.4,1.0),'support':int(rng.integers(10,60))}

def compatibility(a,b,variant):
    return float(models[variant].predict_proba(pair_features(a,b,variant)[None,:])[0,1])

def registry_trial(variant,source_count,quorum,artifact_mode,seed):
    rng=np.random.default_rng(seed); reports=[]
    for activity in CFG.final_novel:
        chosen=rng.choice(CFG.final_subjects,source_count,replace=False)
        reports += [{'client':int(s),'label':activity,'signature':final_candidates[(int(s),activity)]} for s in chosen]
    for subject in CFG.final_subjects:
        for j in range(2): reports.append({'client':subject,'label':f'artifact-{subject}-{j}','signature':synthetic_candidate(subject,artifact_mode,rng)})
    rng.shuffle(reports); entries=[]; ambiguous=0
    for report in reports:
        choices=[]
        for idx,entry in enumerate(entries):
            if report['client'] in {r['client'] for r in entry}: continue
            scores=[compatibility(report['signature'],r['signature'],variant) for r in entry]
            if min(scores)>=thresholds[variant]: choices.append((float(np.mean(scores)),idx))
        choices.sort(reverse=True)
        if choices and (len(choices)==1 or choices[0][0]-choices[1][0]>=CFG.registry_margin):
            entries[choices[0][1]].append(report)
        else:
            if len(choices)>1: ambiguous+=1
            entries.append([report])
    promoted=[e for e in entries if len({r['client'] for r in e})>=quorum]
    correct=[]; labels_recovered=set(); merges=0
    for entry in promoted:
        labels={r['label'] for r in entry}; real={v for v in labels if not v.startswith('artifact-')}; artifacts=labels-real
        if len(real)==1 and not artifacts: correct.append(entry); labels_recovered|=real
        if len(real)>1 or (real and artifacts): merges+=1
    precision=len(correct)/len(promoted) if promoted else np.nan
    return {'precision':precision,'recall':len(labels_recovered)/len(CFG.final_novel),'false_merge_rate':merges/len(promoted) if promoted else 0.0,
            'promoted':len(promoted),'ambiguous_reports':ambiguous}

registry_rows=[]
for variant in ('soft','prototype','dual'):
  for mode in ('independent','mixture'):
    for sources_per_class in (2,4,8):
      for quorum in (2,3,4):
        trials=[registry_trial(variant,sources_per_class,quorum,mode,CFG.seed+trial) for trial in range(CFG.registry_trials)]
        registry_rows.append({'variant':variant,'artifact_mode':mode,'sources_per_class':sources_per_class,'quorum':quorum,
          **{key:float(np.nanmean([t[key] for t in trials])) for key in trials[0]}})
registry_results=pd.DataFrame(registry_rows)
registry_results.sort_values(['artifact_mode','variant','sources_per_class','quorum']).head(20)

,variant,artifact_mode,sources_per_class,quorum,precision,recall,false_merge_rate,promoted,ambiguous_reports
36,dual,independent,2,2,0.009470,0.003333,0.635810,2.05,21.65
37,dual,independent,2,3,0.000000,0.000000,0.460000,0.60,21.65
38,dual,independent,2,4,0.000000,0.000000,0.210000,0.21,21.65
39,dual,independent,4,2,0.046703,0.015000,0.742548,2.30,33.19
40,dual,independent,4,3,0.011111,0.001667,0.440000,0.55,33.19
41,dual,independent,4,4,0.000000,0.000000,0.190000,0.21,33.19
42,dual,independent,8,2,0.079181,0.028333,0.744310,2.34,56.37
43,dual,independent,8,3,0.000000,0.000000,0.580000,0.72,56.37
44,dual,independent,8,4,0.000000,0.000000,0.280000,0.31,56.37
18,prototype,independent,2,2,0.010909,0.010000,0.743532,5.89,13.31


## Decision and saved outputs

The primary registry condition is eight available sources, a three-client quorum, and independent artifacts. Mixture results do not have to meet the same recall target, but abstention must prevent forced high-confidence false promotion.

In [9]:
dual_pair=pair_results.query("variant=='dual'").iloc[0]
primary=registry_results.query("variant=='dual' and artifact_mode=='independent' and sources_per_class==8 and quorum==3").iloc[0]
decision={
    'pair_auroc_pass':bool(dual_pair.test_auroc>=.85),
    'registry_precision_pass':bool(primary.precision>=.90),
    'registry_recall_pass':bool(primary.recall>=.70),
}
decision['proceed_to_full_registry']=all(decision.values())
manifest={'config':asdict(CFG),'device':str(DEVICE),'known_test_accuracy':known_test_accuracy,
          'decision':decision,'timestamp_utc':pd.Timestamp.utcnow().isoformat()}
pair_results.to_csv(ARTIFACTS/'pair_results.csv',index=False)
registry_results.to_csv(ARTIFACTS/'registry_results.csv',index=False)
(ARTIFACTS/'phase0_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
display(pair_results)
display(registry_results.query("variant=='dual' and sources_per_class==8"))

{
  "config": {
    "project": "/content/drive/MyDrive/FedVocabHAR",
    "seed": 42,
    "window": 128,
    "stride": 64,
    "train_fraction": 0.6,
    "known": [
      "A",
      "B",
      "C",
      "D",
      "E",
      "R"
    ],
    "development_novel": [
      "F",
      "G",
      "H",
      "I",
      "K"
    ],
    "final_novel": [
      "L",
      "M",
      "O",
      "P",
      "Q",
      "S"
    ],
    "development_subjects": [
      1635,
      1636,
      1637,
      1638,
      1639,
      1640,
      1641
    ],
    "final_subjects": [
      1643,
      1644,
      1645,
      1646,
      1647,
      1648,
      1649,
      1650
    ],
    "batch_size": 256,
    "epochs": 30,
    "learning_rate": 0.001,
    "embedding_dim": 64,
    "pair_target_fpr": 0.05,
    "registry_margin": 0.05,
    "registry_trials": 100
  },
  "device": "cuda",
  "known_test_accuracy": 0.6359277019549981,
  "decision": {
    "pair_auroc_pass": false,
    "registry_precision_pass": false,
    

,variant,test_auroc,test_average_precision,test_brier,test_ece,threshold,test_tpr,test_fpr
0,soft,0.531222,0.225918,0.495375,0.553270,0.614167,0.559524,0.601190
1,prototype,0.540781,0.170578,0.328829,0.322348,0.873440,0.178571,0.202381
2,dual,0.526396,0.231862,0.653086,0.657124,0.904786,0.636905,0.669048


,variant,artifact_mode,sources_per_class,quorum,precision,recall,false_merge_rate,promoted,ambiguous_reports
42,dual,independent,8,2,0.079181,0.028333,0.744310,2.34,56.37
43,dual,independent,8,3,0.000000,0.000000,0.580000,0.72,56.37
44,dual,independent,8,4,0.000000,0.000000,0.280000,0.31,56.37
51,dual,mixture,8,2,0.088855,0.035000,0.817714,2.85,56.45
52,dual,mixture,8,3,0.000000,0.000000,0.510000,0.74,56.45
53,dual,mixture,8,4,0.000000,0.000000,0.200000,0.22,56.45


## Interpretation rule

- If the dual signature passes all three criteria, freeze its representation and calibration protocol before implementing local novelty detection and integration.
- If prototype-only passes but dual does not, inspect matcher calibration and feature scaling before changing the encoder.
- If neither prototype nor dual reaches AUROC 0.85, improve the reference representation before adding registry complexity.
- If pair discrimination passes but registry precision fails, tighten ambiguity handling, persistence, or complete-link constraints. Increasing quorum alone is not considered a sufficient correction.